<a href="https://colab.research.google.com/github/astikarganesh/25-26-4093-Ganesh-AI-B-DS/blob/main/WEEK-10/Community_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 38) Write Code to Implement Community Networks

This notebook implements comprehensive community network detection and analysis including:
1. Different Community Detection Algorithms
2. Louvain Algorithm (Modularity Optimization)
3. Label Propagation Community Detection
4. K-Clique Percolation
5. Girvan-Newman Algorithm
6. Community Quality Metrics
7. Visualization of Communities

## 1. Import Required Libraries

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from networkx.algorithms import community
from collections import defaultdict, Counter
import seaborn as sns
from itertools import combinations

print("All libraries imported successfully!")

## 2. Create Sample Network for Community Detection

In [ ]:
# Create a network with natural communities
def create_community_network():
    """
    Create a network with clear community structure.
    Three communities densely connected internally,
    sparsely connected between communities.
    """
    G = nx.Graph()
    
    # Community 1: Technology Team
    tech_team = ['Alice', 'Bob', 'Charlie', 'David', 'Eve']
    tech_edges = [
        ('Alice', 'Bob'), ('Alice', 'Charlie'),
        ('Bob', 'Charlie'), ('Bob', 'David'),
        ('Charlie', 'David'), ('David', 'Eve'),
        ('Eve', 'Alice')
    ]
    
    # Community 2: Marketing Team
    marketing_team = ['Frank', 'Grace', 'Henry', 'Iris', 'Jack']
    marketing_edges = [
        ('Frank', 'Grace'), ('Frank', 'Henry'),
        ('Grace', 'Henry'), ('Grace', 'Iris'),
        ('Henry', 'Iris'), ('Iris', 'Jack'),
        ('Jack', 'Frank')
    ]
    
    # Community 3: Sales Team
    sales_team = ['Kate', 'Leo', 'Mike', 'Nora', 'Oscar']
    sales_edges = [
        ('Kate', 'Leo'), ('Kate', 'Mike'),
        ('Leo', 'Mike'), ('Leo', 'Nora'),
        ('Mike', 'Nora'), ('Nora', 'Oscar'),
        ('Oscar', 'Kate')
    ]
    
    # Add all nodes
    all_nodes = tech_team + marketing_team + sales_team
    G.add_nodes_from(all_nodes)
    
    # Add edges within communities
    G.add_edges_from(tech_edges)
    G.add_edges_from(marketing_edges)
    G.add_edges_from(sales_edges)
    
    # Add inter-community edges (bridges)
    bridges = [
        ('Eve', 'Frank'),      # Tech-Marketing bridge
        ('Henry', 'Kate'),     # Marketing-Sales bridge
        ('Oscar', 'Alice')     # Sales-Tech bridge
    ]
    G.add_edges_from(bridges)
    
    return G, {0: set(tech_team), 1: set(marketing_team), 2: set(sales_team)}

G, ground_truth_communities = create_community_network()

print(f"Network created successfully!")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Network density: {nx.density(G):.4f}")
print(f"\nGround truth communities:")
for comm_id, nodes in ground_truth_communities.items():
    print(f"  Community {comm_id + 1}: {nodes}")

## 3. Visualize the Network

In [ ]:
# Visualize the network structure
plt.figure(figsize=(14, 10))

# Create color map based on ground truth communities
color_map = {}
colors = ['#FF6B6B', '#4ECDC4', '#FFE66D']

for comm_id, nodes in ground_truth_communities.items():
    for node in nodes:
        color_map[node] = colors[comm_id]

# Layout
pos = nx.spring_layout(G, k=2, iterations=100, seed=42)

# Draw nodes
node_colors = [color_map[node] for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1500, 
                       edgecolors='black', linewidths=2)

# Draw edges
nx.draw_networkx_edges(G, pos, width=2, alpha=0.6)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')

plt.title("Community Network Structure", fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("Network visualization complete!")

## 4. Greedy Modularity Communities (NetworkX Default)

In [ ]:
print("="*60)
print("1. GREEDY MODULARITY COMMUNITIES")
print("="*60)

# Greedy modularity optimization
greedy_communities = list(community.greedy_modularity_communities(G))
greedy_modularity = community.modularity(G, greedy_communities)

print(f"\nNumber of communities detected: {len(greedy_communities)}")
print(f"Modularity score: {greedy_modularity:.6f}")
print(f"\nCommunities:")
for i, comm in enumerate(greedy_communities):
    print(f"  Community {i+1}: {sorted(list(comm))}")

## 5. Louvain Algorithm (Best Modularity)

In [ ]:
print("\n" + "="*60)
print("2. LOUVAIN ALGORITHM")
print("="*60)

try:
    from networkx.algorithms.community import louvain_communities
    
    # Louvain algorithm
    louvain_communities_result = list(louvain_communities(G, seed=42))
    louvain_modularity = community.modularity(G, louvain_communities_result)
    
    print(f"\nNumber of communities detected: {len(louvain_communities_result)}")
    print(f"Modularity score: {louvain_modularity:.6f}")
    print(f"\nCommunities:")
    for i, comm in enumerate(louvain_communities_result):
        print(f"  Community {i+1}: {sorted(list(comm))}")
except ImportError:
    print("\nLouvain algorithm not available in this NetworkX version.")
    print("Installing python-louvain package would enable this algorithm.")

## 6. Label Propagation Community Detection

In [ ]:
print("\n" + "="*60)
print("3. LABEL PROPAGATION ALGORITHM")
print("="*60)

# Label propagation algorithm
label_prop_communities = list(community.label_propagation_communities(G))
label_prop_modularity = community.modularity(G, label_prop_communities)

print(f"\nNumber of communities detected: {len(label_prop_communities)}")
print(f"Modularity score: {label_prop_modularity:.6f}")
print(f"\nCommunities:")
for i, comm in enumerate(label_prop_communities):
    print(f"  Community {i+1}: {sorted(list(comm))}")

## 7. K-Clique Percolation Method

In [ ]:
print("\n" + "="*60)
print("4. K-CLIQUE PERCOLATION METHOD")
print("="*60)

try:
    # K-clique percolation
    kclique_communities = []
    for k in range(2, 5):
        try:
            cliques = list(community.k_clique_communities(G, k))
            if cliques:
                print(f"\nK={k}:")
                print(f"  Number of communities: {len(cliques)}")
                for i, comm in enumerate(cliques):
                    print(f"  Community {i+1}: {sorted(list(comm))}")
                kclique_communities = cliques
                break
        except:
            continue
except Exception as e:
    print(f"K-clique percolation encountered an issue: {str(e)}")

## 8. Girvan-Newman Algorithm

In [ ]:
print("\n" + "="*60)
print("5. GIRVAN-NEWMAN ALGORITHM")
print("="*60)

def girvan_newman_communities(graph, num_communities=3):
    """
    Girvan-Newman algorithm for community detection.
    Removes edges with highest betweenness centrality iteratively.
    """
    G_copy = graph.copy()
    
    while G_copy.number_of_edges() > 0:
        # Calculate betweenness centrality of all edges
        betweenness = nx.edge_betweenness_centrality(G_copy)
        
        # Remove edge with highest betweenness centrality
        max_betweenness_edge = max(betweenness, key=betweenness.get)
        G_copy.remove_edge(*max_betweenness_edge)
        
        # Get connected components
        components = list(nx.connected_components(G_copy))
        
        # If we have desired number of communities, stop
        if len(components) >= num_communities:
            return components[:num_communities]
    
    return list(nx.connected_components(G_copy))

# Apply Girvan-Newman algorithm
gn_communities = girvan_newman_communities(G, num_communities=3)
gn_modularity = community.modularity(G, gn_communities)

print(f"\nNumber of communities detected: {len(gn_communities)}")
print(f"Modularity score: {gn_modularity:.6f}")
print(f"\nCommunities:")
for i, comm in enumerate(gn_communities):
    print(f"  Community {i+1}: {sorted(list(comm))}")

## 9. Custom Community Detection Algorithm

In [ ]:
print("\n" + "="*60)
print("6. CUSTOM SPECTRAL CLUSTERING FOR COMMUNITIES")
print("="*60)

def spectral_community_detection(graph, num_communities=3):
    """
    Spectral clustering approach using Laplacian matrix.
    """
    from sklearn.cluster import SpectralClustering
    import numpy as np
    
    # Create adjacency matrix
    adj_matrix = nx.to_numpy_array(graph)
    
    # Apply spectral clustering
    sc = SpectralClustering(n_clusters=num_communities, affinity='precomputed',
                            random_state=42, assign_labels='kmeans')
    labels = sc.fit_predict(adj_matrix)
    
    # Convert labels to communities
    nodes = list(graph.nodes())
    communities = [set() for _ in range(num_communities)]
    
    for node, label in zip(nodes, labels):
        communities[label].add(node)
    
    return [comm for comm in communities if comm]  # Remove empty communities

try:
    spectral_communities = spectral_community_detection(G, num_communities=3)
    spectral_modularity = community.modularity(G, spectral_communities)
    
    print(f"\nNumber of communities detected: {len(spectral_communities)}")
    print(f"Modularity score: {spectral_modularity:.6f}")
    print(f"\nCommunities:")
    for i, comm in enumerate(spectral_communities):
        print(f"  Community {i+1}: {sorted(list(comm))}")
except ImportError:
    print("\nSckit-learn is required for spectral clustering.")

## 10. Community Quality Metrics

In [ ]:
print("\n" + "="*60)
print("COMMUNITY QUALITY METRICS")
print("="*60)

def calculate_community_metrics(graph, communities):
    """
    Calculate various metrics for community quality.
    """
    metrics = {}
    
    # 1. Modularity
    modularity = community.modularity(graph, communities)
    metrics['Modularity'] = modularity
    
    # 2. Conductance (community isolation)
    conductances = []
    for comm in communities:
        subgraph = graph.subgraph(comm)
        internal_edges = subgraph.number_of_edges()
        
        # Edges going out of community
        cut_edges = 0
        for node in comm:
            for neighbor in graph.neighbors(node):
                if neighbor not in comm:
                    cut_edges += 1
        
        total_degree = sum(graph.degree(node) for node in comm)
        conductance = cut_edges / total_degree if total_degree > 0 else 0
        conductances.append(conductance)
    
    metrics['Avg Conductance'] = np.mean(conductances) if conductances else 0
    metrics['Min Conductance'] = min(conductances) if conductances else 0
    metrics['Max Conductance'] = max(conductances) if conductances else 0
    
    # 3. Internal Density
    densities = []
    for comm in communities:
        subgraph = graph.subgraph(comm)
        density = nx.density(subgraph)
        densities.append(density)
    
    metrics['Avg Density'] = np.mean(densities) if densities else 0
    
    # 4. Coverage
    all_nodes = set()
    for comm in communities:
        all_nodes.update(comm)
    metrics['Coverage'] = len(all_nodes) / graph.number_of_nodes()
    
    return metrics

# Calculate metrics for different algorithms
all_results = {}

algorithms = {
    'Greedy Modularity': greedy_communities,
    'Label Propagation': label_prop_communities,
    'Girvan-Newman': gn_communities,
}

for algo_name, comms in algorithms.items():
    metrics = calculate_community_metrics(G, comms)
    all_results[algo_name] = metrics
    print(f"\n{algo_name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

## 11. Comparison Table of All Algorithms

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame(all_results).T
print("\n" + "="*80)
print("ALGORITHM COMPARISON TABLE")
print("="*80)
print(comparison_df.to_string())
print("="*80)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Modularity comparison
modularity_scores = [all_results[algo]['Modularity'] for algo in all_results.keys()]
axes[0, 0].bar(all_results.keys(), modularity_scores, color=['#FF6B6B', '#4ECDC4', '#FFE66D'])
axes[0, 0].set_title('Modularity Score Comparison', fontweight='bold')
axes[0, 0].set_ylabel('Modularity')
axes[0, 0].tick_params(axis='x', rotation=45)

# Conductance comparison
conductance_scores = [all_results[algo]['Avg Conductance'] for algo in all_results.keys()]
axes[0, 1].bar(all_results.keys(), conductance_scores, color=['#FF6B6B', '#4ECDC4', '#FFE66D'])
axes[0, 1].set_title('Average Conductance Comparison', fontweight='bold')
axes[0, 1].set_ylabel('Avg Conductance')
axes[0, 1].tick_params(axis='x', rotation=45)

# Density comparison
density_scores = [all_results[algo]['Avg Density'] for algo in all_results.keys()]
axes[1, 0].bar(all_results.keys(), density_scores, color=['#FF6B6B', '#4ECDC4', '#FFE66D'])
axes[1, 0].set_title('Average Density Comparison', fontweight='bold')
axes[1, 0].set_ylabel('Avg Density')
axes[1, 0].tick_params(axis='x', rotation=45)

# Coverage comparison
coverage_scores = [all_results[algo]['Coverage'] for algo in all_results.keys()]
axes[1, 1].bar(all_results.keys(), coverage_scores, color=['#FF6B6B', '#4ECDC4', '#FFE66D'])
axes[1, 1].set_title('Coverage Comparison', fontweight='bold')
axes[1, 1].set_ylabel('Coverage')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 12. Visualization of Communities from Different Algorithms

In [ ]:
# Visualize communities from different algorithms
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

algorithms_viz = {
    'Greedy Modularity': (greedy_communities, axes[0, 0]),
    'Label Propagation': (label_prop_communities, axes[0, 1]),
    'Girvan-Newman': (gn_communities, axes[1, 0]),
    'Ground Truth': (list(ground_truth_communities.values()), axes[1, 1])
}

colors = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#95E1D3', '#F38181']
pos = nx.spring_layout(G, k=2, iterations=100, seed=42)

for algo_name, (comms, ax) in algorithms_viz.items():
    # Create color map for this algorithm
    color_map = {}
    for comm_idx, comm in enumerate(comms):
        for node in comm:
            color_map[node] = colors[comm_idx % len(colors)]
    
    # Draw network
    node_colors = [color_map.get(node, '#CCCCCC') for node in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1000,
                          edgecolors='black', linewidths=1.5, ax=ax)
    nx.draw_networkx_edges(G, pos, width=1.5, alpha=0.5, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)
    
    ax.set_title(algo_name, fontsize=14, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Community visualization from different algorithms complete!")

## 13. Community Structure Analysis

In [ ]:
print("\n" + "="*60)
print("COMMUNITY STRUCTURE ANALYSIS")
print("="*60)

def analyze_community_structure(graph, communities, algo_name):
    """
    Detailed analysis of community structure.
    """
    print(f"\n{algo_name}:")
    print("-" * 60)
    
    for i, comm in enumerate(communities):
        subgraph = graph.subgraph(comm)
        
        print(f"\n  Community {i+1}:")
        print(f"    Nodes: {sorted(list(comm))}")
        print(f"    Size: {len(comm)}")
        print(f"    Internal edges: {subgraph.number_of_edges()}")
        print(f"    Density: {nx.density(subgraph):.4f}")
        
        # Average degree within community
        avg_degree = 2 * subgraph.number_of_edges() / len(comm) if len(comm) > 0 else 0
        print(f"    Avg internal degree: {avg_degree:.2f}")
        
        # External edges
        external_edges = 0
        for node in comm:
            for neighbor in graph.neighbors(node):
                if neighbor not in comm:
                    external_edges += 1
        print(f"    External edges: {external_edges}")

# Analyze best algorithm (Greedy Modularity)
analyze_community_structure(G, greedy_communities, "Greedy Modularity Communities")
analyze_community_structure(G, label_prop_communities, "Label Propagation Communities")
analyze_community_structure(G, gn_communities, "Girvan-Newman Communities")

## 14. Node-to-Community Mapping and Statistics

In [ ]:
# Create detailed mapping table
print("\n" + "="*80)
print("NODE-TO-COMMUNITY MAPPING (Greedy Modularity)")
print("="*80)

# Create node to community mapping
node_community_map = {}
for comm_idx, comm in enumerate(greedy_communities):
    for node in comm:
        node_community_map[node] = comm_idx + 1

# Create DataFrame
node_stats = []
for node in sorted(G.nodes()):
    degree = G.degree(node)
    comm_id = node_community_map.get(node, -1)
    
    # Count internal and external edges
    internal_edges = 0
    external_edges = 0
    
    for neighbor in G.neighbors(node):
        if node_community_map.get(neighbor) == comm_id:
            internal_edges += 1
        else:
            external_edges += 1
    
    node_stats.append({
        'Node': node,
        'Community': comm_id,
        'Degree': degree,
        'Internal Edges': internal_edges,
        'External Edges': external_edges
    })

df_nodes = pd.DataFrame(node_stats)
print(df_nodes.to_string(index=False))
print("="*80)

## 15. Inter-Community and Intra-Community Statistics

In [ ]:
print("\n" + "="*60)
print("INTER-COMMUNITY AND INTRA-COMMUNITY STATISTICS")
print("="*60)

# Count inter-community and intra-community edges
intra_edges = 0
inter_edges = 0

for edge in G.edges():
    node1, node2 = edge
    comm1 = node_community_map[node1]
    comm2 = node_community_map[node2]
    
    if comm1 == comm2:
        intra_edges += 1
    else:
        inter_edges += 1

total_edges = G.number_of_edges()

print(f"\nTotal edges: {total_edges}")
print(f"Intra-community edges: {intra_edges} ({100*intra_edges/total_edges:.1f}%)")
print(f"Inter-community edges: {inter_edges} ({100*inter_edges/total_edges:.1f}%)")

# Community size distribution
print(f"\nCommunity size distribution:")
for i, comm in enumerate(greedy_communities):
    print(f"  Community {i+1}: {len(comm)} nodes")

## 16. Similarity Analysis Between Algorithms

In [ ]:
def jaccard_similarity(comm_set1, comm_set2):
    """
    Calculate Jaccard similarity between two sets of communities.
    """
    # Convert to frozensets for comparison
    set1 = set(frozenset(c) for c in comm_set1)
    set2 = set(frozenset(c) for c in comm_set2)
    
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    
    return intersection / union if union > 0 else 0

print("\n" + "="*60)
print("ALGORITHM SIMILARITY (Jaccard Index)")
print("="*60)

algo_list = [
    ('Greedy Modularity', greedy_communities),
    ('Label Propagation', label_prop_communities),
    ('Girvan-Newman', gn_communities)
]

similarity_matrix = np.zeros((len(algo_list), len(algo_list)))

for i, (name1, comms1) in enumerate(algo_list):
    for j, (name2, comms2) in enumerate(algo_list):
        similarity_matrix[i][j] = jaccard_similarity(comms1, comms2)

# Print similarity matrix
algo_names = [name for name, _ in algo_list]
print("\nSimilarity Matrix:")
print(pd.DataFrame(similarity_matrix, index=algo_names, columns=algo_names).to_string())

# Visualize similarity matrix
plt.figure(figsize=(8, 6))
sns.heatmap(similarity_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=algo_names, yticklabels=algo_names, cbar_kws={'label': 'Similarity'})
plt.title('Algorithm Similarity Matrix (Jaccard Index)', fontweight='bold')
plt.tight_layout()
plt.show()

## 17. Recommendation and Summary

In [ ]:
print("\n" + "="*80)
print("COMMUNITY DETECTION ALGORITHMS - SUMMARY & RECOMMENDATIONS")
print("="*80)

summary = """
1. GREEDY MODULARITY COMMUNITIES
   - Pros: Fast, good modularity, works well for most networks
   - Cons: May not find optimal solution
   - Best for: Large networks, general-purpose community detection
   - Time complexity: O(n² log n) to O(n log² n)

2. LABEL PROPAGATION
   - Pros: Fast, linear complexity, probabilistic approach
   - Cons: Results may vary due to randomness
   - Best for: Large-scale networks, seed-based propagation
   - Time complexity: O(n + m)

3. GIRVAN-NEWMAN
   - Pros: Theoretically sound, finds hierarchical structure
   - Cons: Slow, O(n³) complexity, not suitable for large networks
   - Best for: Small to medium networks, hierarchical analysis
   - Time complexity: O(n³)

4. K-CLIQUE PERCOLATION
   - Pros: Handles overlapping communities
   - Cons: Computationally expensive for large k
   - Best for: Overlapping community structure detection

5. LOUVAIN ALGORITHM
   - Pros: Excellent modularity, very fast
   - Cons: May miss small communities
   - Best for: Large networks, maximizing modularity

SELECTION GUIDE:
- Network size < 1000 nodes: Use Girvan-Newman for hierarchical structure
- Network size 1000-100000: Use Greedy Modularity or Louvain
- Network size > 100000: Use Label Propagation
- Need overlapping communities: Use K-Clique Percolation
- Priority on speed: Use Label Propagation
- Priority on quality: Use Louvain or Greedy Modularity
"""

print(summary)
print("="*80)